Code source: HuggingFace Audio Course - [How to fine-tune an ASR system with the Trainer API](https://huggingface.co/learn/audio-course/en/chapter5/fine-tuning)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install datasets==2.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


In [ ]:
from datasets import load_dataset

afrispeech = load_dataset("tobiolatunji/afrispeech-200", "twi", "train[:5%]", trust_remote_code=True)

Generating train split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1315it [00:00, 43098.34it/s]


Generating validation split: 0 examples [00:00, ? examples/s]


Reading metadata...: 186it [00:00, 4466.37it/s]


Generating test split: 0 examples [00:00, ? examples/s]


Reading metadata...: 58it [00:00, 29190.02it/s]


In [ ]:
print(afrispeech)

DatasetDict({
    train: Dataset({
        features: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration'],
        num_rows: 1315
    })
    validation: Dataset({
        features: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration'],
        num_rows: 186
    })
    test: Dataset({
        features: ['speaker_id', 'path', 'audio_id', 'audio', 'transcript', 'age_group', 'gender', 'accent', 'domain', 'country', 'duration'],
        num_rows: 58
    })
})


In [ ]:
afrispeech = afrispeech.select_columns(["audio", "transcript"])

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="english", task="transcribe"
)

# Preprocessing

In [ ]:
afrispeech["train"].features

{'audio': Audio(sampling_rate=44100, mono=True, decode=True, id=None),
 'transcript': Value(dtype='string', id=None)}

In [ ]:
from datasets import Audio

sampling_rate = processor.feature_extractor.sampling_rate
afrispeech = afrispeech.cast_column("audio", Audio(sampling_rate=sampling_rate))

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    example = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["transcript"],
    )

    # compute input length of audio sample in seconds
    example["input_length"] = len(audio["array"]) / audio["sampling_rate"]

    return example

In [ ]:
afrispeech = afrispeech.map(
    prepare_dataset, remove_columns=afrispeech.column_names["train"], num_proc=1
)

Map:   0%|          | 0/1315 [00:00<?, ? examples/s]

Map:   0%|          | 0/186 [00:00<?, ? examples/s]

Map:   0%|          | 0/58 [00:00<?, ? examples/s]

In [ ]:
max_input_length = 30.0


def is_audio_in_length_range(length):
    return length < max_input_length

In [ ]:
# filter dataset for audios less than max_input_length
afrispeech["train"] = afrispeech["train"].filter(
    is_audio_in_length_range,
    input_columns=["input_length"],
)

Filter:   0%|          | 0/1315 [00:00<?, ? examples/s]

In [ ]:
afrispeech["train"]

Dataset({
    features: ['input_features', 'labels', 'input_length'],
    num_rows: 1306
})

*Training samples reduced from 1315 to 1306 after applying filter*

# Training and Evaluation

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [
            {"input_features": feature["input_features"][0]} for feature in features
        ]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.4 MB/s eta 0:00:00


In [ ]:
import evaluate

metric = evaluate.load("wer")

In [ ]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # compute orthographic wer
    wer_ortho = 100 * metric.compute(predictions=pred_str, references=label_str)

    # compute normalised WER
    pred_str_norm = [normalizer(pred) for pred in pred_str]
    label_str_norm = [normalizer(label) for label in label_str]
    # filtering step to only evaluate the samples that correspond to non-zero references:
    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [
        label_str_norm[i]
        for i in range(len(label_str_norm))
        if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer_ortho": wer_ortho, "wer": wer}

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [ ]:
from functools import partial

# disable cache during training since it's incompatible with gradient checkpointing
model.config.use_cache = False

# set language and task for generation and re-enable cache
model.generate = partial(
    model.generate, language="english", task="transcribe", use_cache=True
)

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-afrispeech",  # name on the HF Hub
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=50,
    max_steps=500,  # increase to 4000 if you have your own GPU or a Colab paid plan
    # gradient_checkpointing=True,
    gradient_checkpointing=False,
    fp16=True,
    fp16_full_eval=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
    # push_to_hub=False,
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=afrispeech["train"],
    eval_dataset=afrispeech["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor,
)

/tmp/ipython-input-250297916.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


# Zero-Shot Evaluation

In [ ]:
print("Calculating zero-shot WER on test set...")

model.eval() # Ensure model is in evaluation mode and set to appropriate precision
model.config.use_cache = True  # Enable cache for faster inference

Calculating zero-shot WER on test set...


In [ ]:
zero_shot_dataset = afrispeech["test"]
print(f"Using test set with {len(zero_shot_dataset)} samples")

Using test set with 58 samples


In [ ]:
all_predictions = []
all_references = []

print("\nRunning inference on test samples...")

for i in range(len(zero_shot_dataset)):
    sample = zero_shot_dataset[i]

    # Convert input features to tensor
    input_array = np.array(sample["input_features"])
    input_tensor = torch.tensor(input_array).float()

    # Reshape to Whisper format: [batch_size=1, n_mels=80, time]
    if input_tensor.dim() == 2:
        if input_tensor.shape[0] == 80:
            input_tensor = input_tensor.unsqueeze(0)
        else:
            input_tensor = input_tensor.transpose(0, 1).unsqueeze(0)

    # Move to GPU if available
    if torch.cuda.is_available():
        input_tensor = input_tensor.to("cuda")

    # Generate transcription
    with torch.no_grad():
        generated_ids = model.generate(
            input_features=input_tensor,
            max_length=225,
            language="english",
            task="transcribe",
            use_cache=True
        )

    # Decode prediction
    prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Decode reference
    label_ids = [id if id != -100 else processor.tokenizer.pad_token_id for id in sample["labels"]]
    reference = processor.tokenizer.decode(label_ids, skip_special_tokens=True)

    all_predictions.append(prediction)
    all_references.append(reference)

    # Print progress
    if (i + 1) % 10 == 0 or (i + 1) == len(zero_shot_dataset):
        print(f"Processed {i + 1}/{len(zero_shot_dataset)} samples")

# Compute WER
wer_ortho = 100 * metric.compute(predictions=all_predictions, references=all_references)

# Compute Normalized WER
normalizer = BasicTextNormalizer()
pred_norm = [normalizer(pred) for pred in all_predictions]
ref_norm = [normalizer(ref) for ref in all_references]

# Filter out empty references
pred_norm_filtered = []
ref_norm_filtered = []
for pred, ref in zip(pred_norm, ref_norm):
    if len(ref) > 0:
        pred_norm_filtered.append(pred)
        ref_norm_filtered.append(ref)

wer_norm = 100 * metric.compute(predictions=pred_norm_filtered, references=ref_norm_filtered)

# Print results
print("\n" + "="*60)
print("ZERO-SHOT (PRE-TRAINED) WHISPER PERFORMANCE")
print("="*60)
print(f"Orthographic WER: {wer_ortho:.2f}%")
print(f"Normalized WER: {wer_norm:.2f}%")
print("="*60)

# Save initial WER values
initial_wer_ortho = wer_ortho
initial_wer_norm = wer_norm

# Reset model to training configuration
model.train()
model.config.use_cache = False
print("\nModel reset to training mode.")

Calculating zero-shot WER on test dataset (58 samples)...
Test dataset size: 58 samples
Sample input_features type: <class 'list'>
Sample input_features length (outer): 1
First element is list with length: 80
First few values of first element: [[-0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186, -0.7379844188690186

# Training

In [ ]:
trainer.train()

In [ ]:
kwargs = {
    "dataset_tags": "tobiolatunji/afrispeech-200",
    "dataset": "AfriSpeech-200",
    "language": "en",
    "model_name": "whisper-small-afrispeech",
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}

In [ ]:
trainer.push_to_hub(**kwargs)

# Demo

In [45]:
from transformers import pipeline

model_id = "naalamle/whisper-small-afrispeech"  # @param
pipe = pipeline("automatic-speech-recognition", model=model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/356 [00:00<?, ?B/s]

Device set to use cuda:0


In [46]:
def transcribe_speech(filepath):
    output = pipe(
        filepath,
        max_new_tokens=256,
        generate_kwargs={
            "task": "transcribe",
            "language": "english",
        },
        chunk_length_s=30,
        batch_size=8,
    )
    return output["text"]

In [47]:
import gradio as gr

demo = gr.Blocks()

mic_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="microphone", type="filepath"),
    outputs=gr.components.Textbox(),
)

file_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="upload", type="filepath"),
    outputs=gr.components.Textbox(),
)

In [48]:
with demo:
    gr.TabbedInterface(
        [mic_transcribe, file_transcribe],
        ["Transcribe Microphone", "Transcribe Audio File"],
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c765490b1aee4cd3bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c765490b1aee4cd3bc.gradio.live
